# Notebook 1 — Export Marqo-FashionCLIP Vision Encoder to ONNX INT8

Exports only the **vision encoder** of Marqo-FashionCLIP (ViT-B/32 backbone).
Text encoder is not needed — we only embed images.

**Output files:**
- `fashion_clip_vision.onnx` — full precision (~170MB)
- `fashion_clip_vision_int8.onnx` — INT8 quantized (~45MB, 2-3x faster on CPU)

**Run on:** Colab T4 (free) or Kaggle GPU. GPU only needed for export speed — inference runs on CPU.

In [ ]:
# ── 1. Install dependencies ──────────────────────────────────────────────────
!pip install -q transformers torch onnx onnxruntime Pillow numpy

In [ ]:
import torch
import numpy as np
from PIL import Image
from transformers import CLIPVisionModel, CLIPProcessor
import onnx
import onnxruntime as ort
import time
import os

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

In [ ]:
# ── 2. Load Marqo-FashionCLIP ─────────────────────────────────────────────────
# Model card: https://huggingface.co/Marqo/marqo-fashionCLIP
MODEL_ID = "Marqo/marqo-fashionCLIP"

print(f"Loading {MODEL_ID} ...")
processor = CLIPProcessor.from_pretrained(MODEL_ID)
vision_model = CLIPVisionModel.from_pretrained(MODEL_ID).to(DEVICE).eval()
print(f"Vision encoder params: {sum(p.numel() for p in vision_model.parameters()):,}")

In [ ]:
# ── 3. Wrap vision encoder for clean ONNX export ─────────────────────────────
class VisionEncoder(torch.nn.Module):
    """Wraps CLIPVisionModel to accept pixel_values tensor and return pooler_output."""
    def __init__(self, clip_vision):
        super().__init__()
        self.vision = clip_vision

    def forward(self, pixel_values: torch.Tensor) -> torch.Tensor:
        # pixel_values: (B, 3, 224, 224) float32, normalised
        outputs = self.vision(pixel_values=pixel_values)
        return outputs.pooler_output  # (B, 512)

wrapper = VisionEncoder(vision_model).to(DEVICE).eval()

# Sanity check — shape should be (1, 512)
dummy = torch.randn(1, 3, 224, 224).to(DEVICE)
with torch.no_grad():
    out = wrapper(dummy)
print(f"Output shape: {out.shape}")

In [ ]:
# ── 4. Export to ONNX (opset 17) ─────────────────────────────────────────────
ONNX_PATH = "fashion_clip_vision.onnx"

dummy_cpu = torch.randn(1, 3, 224, 224)  # export on CPU for portability
wrapper_cpu = VisionEncoder(CLIPVisionModel.from_pretrained(MODEL_ID).eval())

print("Exporting to ONNX ...")
torch.onnx.export(
    wrapper_cpu,
    dummy_cpu,
    ONNX_PATH,
    input_names=["pixel_values"],
    output_names=["embedding"],
    dynamic_axes={
        "pixel_values": {0: "batch_size"},
        "embedding":    {0: "batch_size"},
    },
    opset_version=17,
    do_constant_folding=True,
)

# Validate the exported model
onnx.checker.check_model(ONNX_PATH)
print(f"ONNX model saved and validated: {ONNX_PATH} ({os.path.getsize(ONNX_PATH) / 1e6:.1f} MB)")

In [ ]:
# ── 5. Validate ONNX outputs match PyTorch ────────────────────────────────────
sess = ort.InferenceSession(ONNX_PATH, providers=["CPUExecutionProvider"])

test_input = np.random.randn(1, 3, 224, 224).astype(np.float32)

with torch.no_grad():
    pt_out = wrapper_cpu(torch.from_numpy(test_input)).numpy()

onnx_out = sess.run(["embedding"], {"pixel_values": test_input})[0]

max_diff = np.abs(pt_out - onnx_out).max()
print(f"Max difference PyTorch vs ONNX: {max_diff:.6f}")
assert max_diff < 1e-4, "ONNX output differs too much from PyTorch — check export"

In [ ]:
# ── 6. INT8 dynamic quantization ─────────────────────────────────────────────
from onnxruntime.quantization import quantize_dynamic, QuantType

INT8_PATH = "fashion_clip_vision_int8.onnx"

print("Quantizing to INT8 ...")
quantize_dynamic(
    ONNX_PATH,
    INT8_PATH,
    weight_type=QuantType.QInt8,
)
print(f"INT8 model saved: {INT8_PATH} ({os.path.getsize(INT8_PATH) / 1e6:.1f} MB)")

In [ ]:
# ── 7. Benchmark latency on CPU ───────────────────────────────────────────────
def benchmark(model_path, n_runs=50):
    sess = ort.InferenceSession(model_path, providers=["CPUExecutionProvider"])
    inp = np.random.randn(1, 3, 224, 224).astype(np.float32)
    # warmup
    for _ in range(5):
        sess.run(["embedding"], {"pixel_values": inp})
    # measure
    t0 = time.perf_counter()
    for _ in range(n_runs):
        sess.run(["embedding"], {"pixel_values": inp})
    elapsed = (time.perf_counter() - t0) / n_runs * 1000
    return elapsed

fp32_ms = benchmark(ONNX_PATH)
int8_ms = benchmark(INT8_PATH)

print(f"FP32 ONNX:  {fp32_ms:.1f} ms/image")
print(f"INT8 ONNX:  {int8_ms:.1f} ms/image")
print(f"Speedup:    {fp32_ms / int8_ms:.2f}x")

In [ ]:
# ── 8. Quick end-to-end test with a real image ────────────────────────────────
# Download a sample fashion image to test the full preprocessing pipeline
import urllib.request
from PIL import Image
import io

CLIP_MEAN = np.array([0.48145466, 0.4578275,  0.40821073], dtype=np.float32)
CLIP_STD  = np.array([0.26862954, 0.26130258, 0.27577711], dtype=np.float32)

def preprocess_image(img: Image.Image) -> np.ndarray:
    """Preprocess a PIL image to match CLIPProcessor output."""
    img = img.convert("RGB").resize((224, 224), Image.BICUBIC)
    arr = np.array(img, dtype=np.float32) / 255.0  # (224, 224, 3)
    arr = (arr - CLIP_MEAN) / CLIP_STD
    arr = arr.transpose(2, 0, 1)                    # (3, 224, 224)
    return arr[np.newaxis]                          # (1, 3, 224, 224)

# Use a solid colour patch as smoke test
test_img = Image.new("RGB", (300, 400), color=(100, 150, 200))
inp = preprocess_image(test_img)

sess_int8 = ort.InferenceSession(INT8_PATH, providers=["CPUExecutionProvider"])
emb = sess_int8.run(["embedding"], {"pixel_values": inp})[0]

print(f"Embedding shape: {emb.shape}")
print(f"Embedding norm:  {np.linalg.norm(emb):.4f}")
print("preprocess_image() function is correct — save this for embed_polyvore.ipynb")

In [ ]:
# ── 9. Download model files ───────────────────────────────────────────────────
# If running on Colab, download both files:
try:
    from google.colab import files
    print("Downloading fashion_clip_vision_int8.onnx ...")
    files.download(INT8_PATH)
    print("Downloading fashion_clip_vision.onnx ...")
    files.download(ONNX_PATH)
except ImportError:
    # Kaggle — files are in the working directory, copy to output
    import shutil
    os.makedirs("/kaggle/working/models", exist_ok=True)
    shutil.copy(ONNX_PATH, "/kaggle/working/models/")
    shutil.copy(INT8_PATH, "/kaggle/working/models/")
    print("Files saved to /kaggle/working/models/")
    print("Add as a Kaggle dataset output for use in embed_polyvore.ipynb")